# Day 01 · 부록 — 내 문서(txt/PDF)로 RAG
Day 01 흐름을 **내 실제 파일**에 적용합니다: `my_docs/` 폴더의 txt·PDF를 읽어
**로컬 벡터DB**에 저장하고, **NVIDIA build**(Qwen)가 그 근거로 답합니다. (임베딩·DB는 내 노트북, 생성만 NVIDIA 토큰)

> `my_docs/` 에 파일만 넣으면 됩니다. 비어 있으면 샘플이 자동 생성돼 바로 돌아갑니다.

## 0 · 설치

In [ ]:
%pip install -q sentence-transformers qdrant-client langchain-text-splitters openai pypdf

## 1 · 설정
개인 키(roster의 `DLI_API_KEY`)만 넣으면 됩니다.

In [ ]:
import os, getpass
from openai import OpenAI

# 생성 엔드포인트: NVIDIA build (기본). 강사 DGX로 바꾸려면 아래 줄 주석 해제.
LLM_BASE_URL = os.getenv("QWEN_BASE_URL", "https://integrate.api.nvidia.com/v1")
# LLM_BASE_URL = "http://124.51.229.210:30001/v1"   # ← 강사 DGX 백업
# NVIDIA API 토큰: 환경변수(NVIDIA_API_KEY) 있으면 자동, 없으면 직접 입력(화면 비표시)
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY") or os.getenv("QWEN_API_KEY") or getpass.getpass("NVIDIA API 토큰(nvapi-...) 입력: ")

_c = OpenAI(base_url=LLM_BASE_URL, api_key=NVIDIA_API_KEY)
_av = [m.id for m in _c.models.list().data]
LLM_MODEL = os.getenv("LLM_MODEL", "qwen/qwen3-next-80b-a3b-instruct")
if LLM_MODEL not in _av:
    _q = [m for m in _av if m.startswith("qwen/") and not any(x in m for x in ("vl", "embed", "rerank"))]
    LLM_MODEL = _q[0] if _q else _av[0]
print("엔드포인트:", LLM_BASE_URL, "| 모델:", LLM_MODEL)

EMBED_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"
DOCS_DIR = "./my_docs"      # 여기에 내 txt/PDF 넣기
DB_PATH  = "./my_qdrant"    # 로컬 벡터DB (디스크에 저장 → 재사용)
REBUILD  = False            # True면 DB를 다시 만듦(문서 바뀌었을 때)

## 2 · 내 문서 불러오기 (txt · PDF)
폴더가 비어 있으면 샘플 파일을 만들어 바로 실행되게 합니다.

In [ ]:
from pathlib import Path
from pypdf import PdfReader

Path(DOCS_DIR).mkdir(exist_ok=True)
if not any(Path(DOCS_DIR).iterdir()):   # 비어 있으면 샘플 생성
    (Path(DOCS_DIR)/"샘플_정책.txt").write_text(
        "연차 휴가는 사용 3영업일 전까지 사내포털에서 신청하고 팀장 승인을 받는다.\n"
        "비용 정산은 영수증 첨부 후 재무팀이 최종 승인한다.\n"
        "재택근무는 주 2회까지 가능하며 전날 18시까지 팀 채널에 공유한다.", encoding="utf-8")
    print("my_docs/ 가 비어 있어 샘플_정책.txt 를 만들었습니다. 내 파일로 교체하세요.")

def load_documents(folder):
    docs = []
    for p in sorted(Path(folder).glob("**/*")):
        if p.suffix.lower() == ".txt":
            docs.append((p.name, p.read_text(encoding="utf-8", errors="ignore")))
        elif p.suffix.lower() == ".pdf":
            text = "\n".join((pg.extract_text() or "") for pg in PdfReader(str(p)).pages)
            if text.strip():
                docs.append((p.name, text))
    return docs

documents = load_documents(DOCS_DIR)
print(f"불러온 문서 {len(documents)}개:", [n for n, _ in documents])

## 3 · 청킹
긴 문서를 겹치는 조각으로 나눕니다. 각 조각에 **출처 파일명**을 붙여 인용에 씁니다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)

chunks = []   # (text, source)
for name, text in documents:
    for c in splitter.split_text(text):
        chunks.append((c, name))
print(len(chunks), "청크")

## 4 · 임베딩 + 로컬 벡터DB 저장 (재사용 가능)
`REBUILD=False`면 이미 만든 DB를 재사용(재임베딩 안 함).

In [ ]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

embedder = SentenceTransformer(EMBED_MODEL)
def embed(xs): return embedder.encode(xs, normalize_embeddings=True)

client = QdrantClient(path=DB_PATH)
exists = client.collection_exists("mydocs") and client.count("mydocs").count > 0

if REBUILD or not exists:
    vecs = embed([c for c, _ in chunks])
    client.recreate_collection("mydocs",
        vectors_config=VectorParams(size=vecs.shape[1], distance=Distance.COSINE))
    client.upsert("mydocs", points=[
        PointStruct(id=i, vector=v.tolist(), payload={"text": chunks[i][0], "source": chunks[i][1]})
        for i, v in enumerate(vecs)])
    print("벡터DB 새로 생성:", client.count("mydocs").count, "청크")
else:
    print("기존 벡터DB 재사용:", client.count("mydocs").count, "청크  (REBUILD=True로 다시 만들 수 있음)")

## 5 · 검색 + 생성
로컬에서 top-k 검색 → 근거를 Qwen에 넘겨 **출처와 함께** 답변.

In [ ]:
from openai import OpenAI
llm = OpenAI(base_url=LLM_BASE_URL, api_key=NVIDIA_API_KEY)   # NVIDIA build (Qwen)

SYSTEM = ("아래 [자료]의 내용만 근거로 답하라. 자료에 없으면 '문서에 없음'이라고만 답하라. "
          "답 끝에 (출처: 파일명)을 표기하라. 추측 금지.")

def search(query, k=4):
    qv = embed([query])[0]
    pts = client.query_points("mydocs", query=qv.tolist(), limit=k).points
    return [(p.payload["text"], p.payload["source"], round(p.score, 3)) for p in pts]

def rag_answer(query, k=4):
    hits = search(query, k)
    ctx = "\n".join(f"[{i+1}] ({src}) {t}" for i, (t, src, _) in enumerate(hits))
    out = llm.chat.completions.create(model=LLM_MODEL, temperature=0.2, max_tokens=300,
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": f"[자료]\n{ctx}\n\n[질문] {query}"}])
    return out.choices[0].message.content

## 6 · 사용 예시
질문을 내 문서에 맞게 바꿔 보세요.

In [ ]:
for q in ["연차는 며칠 전에 신청하나요?", "사내 헬스장 운영시간은?"]:   # 2번째는 문서에 없음 → 거절 확인
    print("Q:", q)
    print("A:", rag_answer(q))
    print("-" * 50)

## 산출물 & 팁
- ✅ `my_docs/` 에 **내 txt·PDF** 넣기 → 로컬 벡터DB → Qwen이 근거로 답변(+출처)
- 💾 `DB_PATH` 에 DB가 **디스크로 저장** → 다음에 재임베딩 없이 재사용. 문서 바꾸면 `REBUILD=True`
- 🔒 원문 전체는 내 노트북을 안 떠나고, **검색된 조각만** Qwen으로 전송
- 🔎 검색이 근거를 못 찾으면 모델은 **지어내지 않고 '문서에 없음'** → 품질이 아쉬우면 Day02(하이브리드+리랭커) 적용
- 📄 스캔(이미지) PDF는 텍스트 추출이 안 됩니다(OCR 필요). 텍스트 기반 PDF·txt 권장